In [ ]:
import requests
import pandas as pd
import pymysql
import random


# API URL (ACCIDENT BASE)

url = # enter your API URL here


# FETCH DATA

data = requests.get(url).json()
df = pd.DataFrame(data)

print("API Loaded")


# SELECT COLUMNS

df = df[[
    "crash_date",
    "crash_time",
    "borough",
    "latitude",
    "longitude",
    "number_of_persons_injured",
    "number_of_persons_killed",
    "contributing_factor_vehicle_1"
]]

df.columns = [
    "date","time","area","latitude","longitude",
    "injured","killed","factor"
]


# CLEAN DATA
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["time"] = pd.to_datetime(df["time"], errors="coerce")

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

df["injured"] = pd.to_numeric(df["injured"], errors="coerce").fillna(0)
df["killed"] = pd.to_numeric(df["killed"], errors="coerce").fillna(0)

df["area"] = df["area"].fillna("Unknown")
df["factor"] = df["factor"].fillna("Unknown")

# REMOVE BAD DATA
df = df.dropna(subset=["date","time","latitude","longitude"])
df = df[df["area"] != "Unknown"]


# TAKE LATEST DATA ONLY

df = df.sort_values("date", ascending=False).head(1000)


# FEATURE ENGINEERING

df["hour"] = df["time"].dt.hour
df["day_name"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month

def peak(h):
    return "Yes" if 7 <= h <= 10 or 17 <= h <= 20 else "No"

def severity(row):
    if row["killed"] > 0:
        return "High"
    elif row["injured"] > 2:
        return "Medium"
    else:
        return "Low"

df["peak_hour"] = df["hour"].apply(peak)
df["severity"] = df.apply(severity, axis=1)



# Different speed per row
df["avg_speed"] = [random.uniform(15, 70) for _ in range(len(df))]

# Travel time varies
df["travel_time"] = [random.uniform(20, 100) for _ in range(len(df))]

# Congestion logic
def congestion(speed):
    if speed < 20:
        return "High"
    elif speed < 40:
        return "Medium"
    else:
        return "Low"

df["congestion_level"] = df["avg_speed"].apply(congestion)

# Infrastructure (random realistic variation)
df["incident_reports"] = [random.randint(1, 50) for _ in range(len(df))]
df["signal_issue"] = [random.randint(0, 10) for _ in range(len(df))]
df["road_issue"] = [random.randint(0, 10) for _ in range(len(df))]
df["cctv_count"] = [random.randint(10, 100) for _ in range(len(df))]

df["accident_count"] = 1


# MYSQL CONNECTION

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="2711",
    database="smart_city"
)

cursor = conn.cursor()

# INSERT QUERY

query = """
INSERT INTO traffic_nyc_final_api (
area, latitude, longitude,
date, time,
hour, day_name, month,
accident_count, injured, killed,
factor,
avg_speed, travel_time, congestion_level,
incident_reports, signal_issue, road_issue, cctv_count,
peak_hour, severity
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""


# INSERT DATA

data_to_insert = []

for _, row in df.iterrows():

    values = (
        row["area"],
        float(row["latitude"]),
        float(row["longitude"]),
        row["date"].date(),
        row["time"].time(),
        int(row["hour"]),
        row["day_name"],
        int(row["month"]),
        1,
        int(row["injured"]),
        int(row["killed"]),
        row["factor"],
        float(row["avg_speed"]),
        float(row["travel_time"]),
        row["congestion_level"],
        int(row["incident_reports"]),
        int(row["signal_issue"]),
        int(row["road_issue"]),
        int(row["cctv_count"]),
        row["peak_hour"],
        row["severity"]
    )

    data_to_insert.append(values)

cursor.executemany(query, data_to_insert)
conn.commit()

print("Inserted rows:", len(data_to_insert))

cursor.close()
conn.close()

API Loaded


C:\Users\ratna\AppData\Local\Temp\ipykernel_21968\1677297748.py:42: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["time"] = pd.to_datetime(df["time"], errors="coerce")


Inserted rows: 1000
